In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import bitsandbytes as bnb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
import evaluate
from peft import LoraConfig, get_peft_model, TaskType
from collections import Counter
from transformers import BitsAndBytesConfig
from collections import Counter

from dotenv import load_dotenv
import os
load_dotenv()
YOUR_HF_TOKEN = os.getenv("YOUR_HF_TOKEN")

/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def combine_text(row):
    # Collect all existing description parts in order
    desc_parts = [row.get(f"case_description_{i}", "") for i in range(1, 9) if row.get(f"case_description_{i}")]
    # Collect all existing justification parts in order
    just_parts = [row.get(f"justification_{i}", "") for i in range(1, 5) if row.get(f"justification_{i}")]
    
    # Prioritize description by putting it first
    return " ".join(desc_parts + just_parts)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
with open("data/regional-court-data.json", 'r', encoding='utf-8') as f:
    data = json.load(f)

if isinstance(data, list):
    labels = [item["label"] for item in data if "label" in item]
    
    label_counts = Counter(labels)
    
    print("Number of samples per label:")
    for label, count in sorted(label_counts.items()):
        print(f"{label:12} : {count:4d} samples")
    
    print(f"\nTotal samples in Regional Court Data: {len(data)}")

Number of samples per label:
nevinovat    : 3626 samples
vinovat      : 5244 samples

Total samples in Regional Court Data: 8870


In [7]:
# Define your mapping
path = "data/regional-court-data.json"
label_map = {
    "vinovat": 0,
    "nevinovat": 1
}

def prepare_dataset(json_file_path):
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    formatted_data = []
    for entry in data:
        # Combine description (1-8) and justification (1-4)
        desc = " ".join([entry.get(f"case_description_{i}", "") for i in range(1, 9)]).strip()
        # just = " ".join([entry.get(f"justification_{i}", "") for i in range(1, 5)]).strip()
        
        formatted_data.append({
            "text": f"{desc}", # Description prioritized by order
            "label": label_map[entry["label"]]
        })
    
    # Split into train and validation (85/15)
    train_data, val_data = train_test_split(
        formatted_data, 
        test_size=0.15, 
        stratify=[d["label"] for d in formatted_data], # Keep class ratios same
        random_state=42
    )
    
    return Dataset.from_list(train_data), Dataset.from_list(val_data)

train_raw, val_raw = prepare_dataset(path)

In [6]:
checkpoint_path_rogpt2 = "results/good/dist-ro-gpt2-base/checkpoint-1416"
rogpt2 = AutoModelForSequenceClassification.from_pretrained(checkpoint_path_rogpt2).to(device)
rogpt2.eval()
print(f"Model loaded from {checkpoint_path_rogpt2}")

checkpoint_path_robertbase = "results/good/dist-ro-bert-base/checkpoint-1888"
robertbase = AutoModelForSequenceClassification.from_pretrained(checkpoint_path_robertbase).to(device)
robertbase.eval()
print(f"Model loaded from {checkpoint_path_robertbase}")

checkpoint_path_legalbert = "results/good/dist-legal-bert/checkpoint-1416"
legalbert = AutoModelForSequenceClassification.from_pretrained(checkpoint_path_legalbert).to(device)
legalbert.eval()
print(f"Model loaded from {checkpoint_path_legalbert}")

Model loaded from results/good/dist-ro-gpt2-base/checkpoint-1416
Model loaded from results/good/dist-ro-bert-base/checkpoint-1888
Model loaded from results/good/dist-legal-bert/checkpoint-1416


In [9]:
import torch
import numpy as np
from tqdm import tqdm
from scipy.stats import mode
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score

# 1. Ensure tokenizers match their respective models
# Note: Ensure you load the correct tokenizers for each architecture type
from transformers import AutoTokenizer
tokenizer_gpt2 = AutoTokenizer.from_pretrained("readerbench/RoGPT2-base")
tokenizer_gpt2.pad_token = tokenizer_gpt2.eos_token
tokenizer_gpt2.padding_side = "right"

tokenizer_bert = AutoTokenizer.from_pretrained("dumitrescustefan/bert-base-romanian-cased-v1")
tokenizer_legal = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased") # or your exact legalbert base repo

y_true = []
gpt2_preds = []
robert_preds = []
legal_preds = []

print("Running ensemble inference over the validation set...")
# Loop directly through your raw validation dataset
with torch.no_grad():
    for item in tqdm(val_raw):
        text = item["text"]
        label = int(item["label"])
        y_true.append(label)
        
        # --- Model 1: RoGPT2 Inference ---
        inputs_gpt2 = tokenizer_gpt2(text, truncation=True, padding="max_length", max_length=512, return_tensors="pt").to(device)
        outputs_gpt2 = rogpt2(**inputs_gpt2)
        pred_gpt2 = torch.argmax(outputs_gpt2.logits, dim=-1).item()
        gpt2_preds.append(pred_gpt2)
        
        # --- Model 2: RoBERTBase Inference ---
        inputs_bert = tokenizer_bert(text, truncation=True, padding="max_length", max_length=512, return_tensors="pt").to(device)
        outputs_bert = robertbase(**inputs_bert)
        pred_bert = torch.argmax(outputs_bert.logits, dim=-1).item()
        robert_preds.append(pred_bert)
        
        # --- Model 3: LegalBERT Inference ---
        inputs_legal = tokenizer_legal(text, truncation=True, padding="max_length", max_length=512, return_tensors="pt").to(device)
        outputs_legal = legalbert(**inputs_legal)
        pred_legal = torch.argmax(outputs_legal.logits, dim=-1).item()
        legal_preds.append(pred_legal)

# 2. Compile predictions into a matrix (Samples x Models)
voting_matrix = np.column_stack((gpt2_preds, robert_preds, legal_preds))

# 3. Apply Majority Voting Strategy
ensemble_preds, _ = mode(voting_matrix, axis=1)
ensemble_preds = ensemble_preds.ravel()

# 4. Calculate Final Metrics
print("\n" + "="*40)
print("INDIVIDUAL MODEL ACCURACIES:")
print(f"RoGPT2 Accuracy:     {accuracy_score(y_true, gpt2_preds):.4f}")
print(f"RoBERTBase Accuracy: {accuracy_score(y_true, robert_preds):.4f}")
print(f"LegalBERT Accuracy:  {accuracy_score(y_true, legal_preds):.4f}")
print("="*40)
print("FINAL ENSEMBLE METRICS (MAJORITY VOTING):")
print(f"Ensemble Accuracy:  {accuracy_score(y_true, ensemble_preds):.4f}")
print(f"Ensemble F1-Macro: {f1_score(y_true, ensemble_preds, average='macro'):.4f}")
print(f"Ensemble Precision:{precision_score(y_true, ensemble_preds, average='macro'):.4f}")
print(f"Ensemble Recall:   {recall_score(y_true, ensemble_preds, average='macro'):.4f}")
print("="*40)

# Full breakdown report for your thesis appendix
print("\nDetailed Classification Report:")
print(classification_report(y_true, ensemble_preds, target_names=["Clasa 0", "Clasa 1"]))

Running ensemble inference over the validation set...


100%|██████████| 1331/1331 [01:20<00:00, 16.58it/s]


INDIVIDUAL MODEL ACCURACIES:
RoGPT2 Accuracy:     0.7889
RoBERTBase Accuracy: 0.8077
LegalBERT Accuracy:  0.7866
FINAL ENSEMBLE METRICS (MAJORITY VOTING):
Ensemble Accuracy:  0.8174
Ensemble F1-Macro: 0.8075
Ensemble Precision:0.8159
Ensemble Recall:   0.8028

Detailed Classification Report:
              precision    recall  f1-score   support

     Clasa 0       0.82      0.88      0.85       787
     Clasa 1       0.81      0.72      0.76       544

    accuracy                           0.82      1331
   macro avg       0.82      0.80      0.81      1331
weighted avg       0.82      0.82      0.82      1331



In [ ]:
# Running ensemble inference over the validation set...
# 100%|██████████| 1331/1331 [01:09<00:00, 19.13it/s]

# ========================================
# INDIVIDUAL MODEL ACCURACIES:
# RoGPT2 Accuracy:     0.7506
# RoBERTBase Accuracy: 0.8077
# LegalBERT Accuracy:  0.7866
# ========================================
# FINAL ENSEMBLE METRICS (MAJORITY VOTING):
# Ensemble Accuracy:  0.8077
# Ensemble F1-Macro: 0.7948
# Ensemble Precision:0.8100
# Ensemble Recall:   0.7883
# ========================================

# Detailed Classification Report:
#               precision    recall  f1-score   support

#      Clasa 0       0.80      0.89      0.85       787
#      Clasa 1       0.82      0.68      0.74       544

#     accuracy                           0.81      1331
#    macro avg       0.81      0.79      0.79      1331
# weighted avg       0.81      0.81      0.80      1331

